# Data Validation (Great Expectation)

# 1. Import Libraries

In [28]:
from pathlib import Path
import json
import pandas as pd
import great_expectations as gx
from great_expectations.data_context import FileDataContext

print("GX version:", gx.__version__)
assert gx.__version__ == "0.18.19", "Gunakan Great Expectations 0.18.19."


GX version: 0.18.19


# 2. Load Dataset (cleaned)

In [29]:
df = pd.read_csv('P2M3_Amanda_Amalia_data_clean.csv')
print("Data Shape:", df.shape)
display(df.head())
# Remove unwanted index column
df = df.drop(columns=["unnamed_0"], errors="ignore")

# Check columns
print(df.columns.tolist())


Data Shape: (3900, 19)


,unnamed_0,customer_id,age,gender,item_purchased,category,purchase_amount_usd,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,promo_code_used,previous_purchases,payment_method,frequency_of_purchases
0,0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


['customer_id', 'age', 'gender', 'item_purchased', 'category', 'purchase_amount_usd', 'location', 'size', 'color', 'season', 'review_rating', 'subscription_status', 'shipping_type', 'discount_applied', 'promo_code_used', 'previous_purchases', 'payment_method', 'frequency_of_purchases']


# 3. Data Context, Data Source, Batch Request, and Expectation Suite

In [30]:
# Data context
context = FileDataContext.create(project_root_dir=str(Path.cwd()))
context.add_or_update_expectation_suite(
    expectation_suite_name="shopping_trends_suite",
    expectations=[]
)
# Data Source
datasource = context.sources.add_or_update_pandas(name="shopping_trends_pandas")
# Data Asset
asset_name = "shopping_clean"
if asset_name in [asset.name for asset in datasource.assets]:
    asset = datasource.get_asset(asset_name)
else:
    asset = datasource.add_dataframe_asset(name=asset_name)
# Batch Request
batch_request = asset.build_batch_request(dataframe=df)
# Expectation Suite (validator)
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name="shopping_trends_suite"
)
validator.head()

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

,customer_id,age,gender,item_purchased,category,purchase_amount_usd,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,promo_code_used,previous_purchases,payment_method,frequency_of_purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   customer_id             3900 non-null   int64  
 1   age                     3900 non-null   int64  
 2   gender                  3900 non-null   object 
 3   item_purchased          3900 non-null   object 
 4   category                3900 non-null   object 
 5   purchase_amount_usd     3900 non-null   int64  
 6   location                3900 non-null   object 
 7   size                    3900 non-null   object 
 8   color                   3900 non-null   object 
 9   season                  3900 non-null   object 
 10  review_rating           3900 non-null   float64
 11  subscription_status     3900 non-null   object 
 12  shipping_type           3900 non-null   object 
 13  discount_applied        3900 non-null   object 
 14  promo_code_used         3900 non-null   

# 4. To be unique (Expectation 1)

In [32]:
result_1 = validator.expect_column_values_to_not_be_null(column="customer_id")

result_1


Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 3900,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

# 5. to be between min_value and max_value (Expectation 2)

In [33]:
result_2 = validator.expect_column_values_to_be_between(column="review_rating", min_value=1, max_value=5)

result_2


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 3900,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

# 6. to be in set (Expectation 3)

In [34]:
result_3 = validator.expect_column_values_to_be_in_set(column="category", value_set=["Clothing", "Accessories", "Footwear", "Outerwear"])

result_3


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 3900,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

# 7. to be in type list (Expectation 4)

In [35]:
result_4 = validator.expect_column_values_to_be_in_type_list(column="purchase_amount_usd", type_list=["int", "int64", "float", "float64"])

result_4


Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "observed_value": "int64"
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

# 8. Ordered column list (Expectation 5)

In [36]:
result_5 = validator.expect_table_columns_to_match_ordered_list(column_list=[
    'customer_id', 'age', 'gender', 'item_purchased', 'category', 'purchase_amount_usd', 'location', 'size', 'color', 'season', 'review_rating', 'subscription_status', 'shipping_type', 'discount_applied', 'promo_code_used', 'previous_purchases', 'payment_method', 'frequency_of_purchases'
    ])

result_5


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "observed_value": [
      "customer_id",
      "age",
      "gender",
      "item_purchased",
      "category",
      "purchase_amount_usd",
      "location",
      "size",
      "color",
      "season",
      "review_rating",
      "subscription_status",
      "shipping_type",
      "discount_applied",
      "promo_code_used",
      "previous_purchases",
      "payment_method",
      "frequency_of_purchases"
    ]
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

# 9. Row count equal (Expectation 6) 

In [37]:
result_6 = validator.expect_table_row_count_to_equal(value=3900)

result_6


Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "observed_value": 3900
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

# 10. Match regex — item_purchased (Expectation 7)

In [38]:
result_7 = validator.expect_column_values_to_match_regex(column="item_purchased", regex=r"^[A-Za-z]+(?:[ -][A-Za-z]+)*$")

result_7


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 3900,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

# 12. save Suite, Run Checkpoint, Create Data Docs

In [39]:
# Save into Expectation Suite 
validator.save_expectation_suite(discard_failed_expectations=False)
# Create a checkpoint
checkpoint = context.add_or_update_checkpoint(
    name="shopping_trends_checkpoint",
    validator=validator
)
# Run Checkpoint
checkpoint_result = checkpoint.run()

print("Checkpoint success:", checkpoint_result.success)

# Build Data Docs
context.build_data_docs()

# Open Data Docs
context.build_data_docs()

Calculating Metrics:   0%|          | 0/29 [00:00<?, ?it/s]

Checkpoint success: True


{'local_site': 'file:///Users/amandaamalia/P2-M3-FTDS-044-HCK-amanda1219004/gx/uncommitted/data_docs/local_site/index.html'}

In [40]:
checkpoint_result = checkpoint.run()

print("Checkpoint success:", checkpoint_result.success)

# Lihat hasil validasi
for run_result in checkpoint_result.run_results.values():
    validation_result = run_result["validation_result"]

    for result in validation_result.results:
        if not result.success:
            print("FAILED:", result.expectation_config.expectation_type)
            print("Column:", result.expectation_config.kwargs.get("column"))
            print("Details:", result.result)
            print("-" * 50)

Calculating Metrics:   0%|          | 0/29 [00:00<?, ?it/s]

Checkpoint success: True
